# TinyMyo (8-layer) EMG-to-text — verify + compress
Reproduce the author's fine-tuned checkpoint (~34.69% WER), export to ONNX, apply dynamic INT8, and measure WER + size.
Use the **Python 3.11 runtime**. Edit `DRIVE_PROJECT` in Cell 3 if your folder differs.

In [1]:
!nvidia-smi
import sys, torch
print("Python:", sys.version.split()[0], "| Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Thu Sep 17 11:51:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip install -q onnx onnxruntime timm jiwer unidecode omegaconf praat-textgrids \
    librosa speechbrain torchinfo torchprofile h5py transformers flashlight-text huggingface_hub pyyaml
from torchaudio.models.decoder import ctc_decoder
print("torchaudio ctc_decoder import OK")

torchaudio ctc_decoder import OK


In [3]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, shutil
# ===================== EDIT THIS =====================
DRIVE_PROJECT = "/content/drive/MyDrive/silent_speech"   # your project folder on Drive
# ====================================================
REPO_DIR = "/content/silent_speech"
LM_DIR   = os.path.join(REPO_DIR, "KenLM")
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/MatteoFasulo/silent_speech.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
!sed -i '/norm_layer=norm_layer,/d' {REPO_DIR}/architecture.py
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
cwd: /content/silent_speech


In [4]:
from huggingface_hub import hf_hub_download
# If the repo is gated: from huggingface_hub import login; login("hf_xxx")  # then re-run
src = hf_hub_download(repo_id="PulpBio/TinyMyo",
                      filename="Silent_Speech/emg-to-text/tinymyo_ft_emg2text_epoch_157.pt")
CKPT = os.path.join(REPO_DIR, "tinymyo_ft_emg2text_epoch_157.pt")
shutil.copy(src, CKPT)
print("checkpoint:", CKPT, f"{os.path.getsize(CKPT)/1e6:.1f} MB")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


checkpoint: /content/silent_speech/tinymyo_ft_emg2text_epoch_157.pt 18.3 MB


In [5]:
import glob, yaml
# --- dataset: patch the YAML config h5_path to your real file ---
H5_REAL = f"{DRIVE_PROJECT}/emg_dataset.h5"
if not os.path.exists(H5_REAL):
    c = sorted(glob.glob("/content/drive/MyDrive/**/emg_dataset.h5", recursive=True))
    assert c, "emg_dataset.h5 not found on Drive"
    H5_REAL = c[0]
os.environ["CKPT_DIR"]  = "/content/ckpts"; os.makedirs("/content/ckpts", exist_ok=True)
os.environ["DATA_PATH"] = DRIVE_PROJECT
patched = []
for cfg in glob.glob(os.path.join(REPO_DIR, "config", "*.y*ml")):
    with open(cfg) as f:
        d = yaml.safe_load(f) or {}
    if isinstance(d, dict) and "h5_path" in d:
        d["h5_path"] = H5_REAL
        with open(cfg, "w") as f:
            yaml.safe_dump(d, f, sort_keys=False)
        patched.append(os.path.basename(cfg))
print("h5:", H5_REAL, "| patched:", patched)

# --- KenLM: repo ships gaddy_derived_lexicon.txt; copy lm.bin in from Drive ---
os.makedirs(LM_DIR, exist_ok=True)
lm = glob.glob("/content/drive/MyDrive/**/lm.bin", recursive=True) + glob.glob(f"{REPO_DIR}/**/lm.bin", recursive=True)
assert lm, "lm.bin not found on Drive or repo."
dst = os.path.join(LM_DIR, "lm.bin")
if os.path.abspath(lm[0]) != os.path.abspath(dst):
    shutil.copy(lm[0], dst)
print("KenLM dir:", os.listdir(LM_DIR))

h5: /content/drive/MyDrive/silent_speech/emg_dataset.h5 | patched: ['data.yaml']
KenLM dir: ['gaddy_derived_lexicon.txt', 'lm.bin']


In [6]:
!cd {REPO_DIR} && CKPT_DIR=/content/ckpts DATA_PATH={DRIVE_PROJECT} \
    python recognition_model.py --model tinymyo --evaluate_saved {CKPT}

2026-09-17 11:52:01.078141: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-17 11:52:01.096320: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789645921.118384    9463 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789645921.125664    9463 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-17 11:52:01.147466: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [7]:
import numpy as np, torch.nn.functional as F, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from hdf5_dataset import H5EmgDataset
from architecture import EMGTransformer

testset = H5EmgDataset(dev=False, test=True)
print("sample keys:", list(testset[0].keys()))
n_chars = len(testset.text_transform.chars)
model = EMGTransformer(num_features=testset.num_features, num_outs=n_chars + 1,
                       in_chans=8, embed_dim=192, n_layer=8, n_head=3, mlp_ratio=4.0,
                       attn_drop=0.1, proj_drop=0.1).eval()
sd = torch.load(CKPT, map_location="cpu", weights_only=False)
if isinstance(sd, dict) and "model_state_dict" in sd:
    sd = sd["model_state_dict"]
model.load_state_dict(sd, strict=True)
print("params:", sum(p.numel() for p in model.parameters()))

class Wrap(torch.nn.Module):
    def __init__(s, m):
        super().__init__(); s.m = m
    def forward(s, x):
        return s.m(x, x, x)

FP32 = "/content/model_fp32_tinymyo.onnx"; INT8 = "/content/model_int8_tinymyo.onnx"
ex = testset[0]["raw_emg"].unsqueeze(0)
torch.onnx.export(Wrap(model).eval(), ex, FP32, input_names=["emg"], output_names=["logits"],
                  dynamic_axes={"emg": {0: "batch", 1: "time"}, "logits": {0: "batch", 1: "time"}},
                  opset_version=17, dynamo=False)
quantize_dynamic(FP32, INT8, weight_type=QuantType.QInt8)
mb = lambda p: os.path.getsize(p) / 1e6
print(f"FP32 {mb(FP32):.2f} MB -> INT8 {mb(INT8):.2f} MB ({mb(FP32)/mb(INT8):.2f}x)")

sample keys: ['audio_features', 'emg', 'raw_emg', 'phonemes', 'text', 'text_int', 'text_int_lengths', 'session_ids', 'book_location', 'silent', 'parallel_voiced_audio_features', 'parallel_voiced_emg', 'audio_file']
params: 4546982


/content/silent_speech/architecture.py:96: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  scale_factor = 1 / math.sqrt(q.size(-1))
/content/silent_speech/transformer_gaddy.py:50: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  pad_length = max(length - self.max_relative_pos, 0)
/content/silent_speech/transformer_gaddy.py:51: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not

FP32 18.97 MB -> INT8 6.69 MB (2.84x)


In [ ]:
import tqdm, jiwer
from torchaudio.models.decoder import ctc_decoder

def build_decoder(dset, beam=1500):
    tkns = [c for c in dset.text_transform.chars] + ["_"]
    return ctc_decoder(lexicon=os.path.join(LM_DIR, "gaddy_derived_lexicon.txt"), tokens=tkns,
                       lm=os.path.join(LM_DIR, "lm.bin"), blank_token="_", sil_token="|",
                       nbest=1, lm_weight=2, beam_size=beam)

def onnx_wer(path, dset, decoder):
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"]); nm = s.get_inputs()[0].name
    refs, preds = [], []
    for i in tqdm.tqdm(range(len(dset)), desc=os.path.basename(path)):
        ex = dset[i]
        x = ex["raw_emg"].numpy().astype(np.float32)[None, ...]
        y = s.run(None, {nm: x})[0]
        logp = F.log_softmax(torch.from_numpy(y), dim=-1)
        pred = dset.text_transform.clean_text(" ".join(decoder(logp)[0][0].words).strip())
        tgt = dset.text_transform.clean_text(ex["text"])
        if tgt != "":
            refs.append(tgt); preds.append(pred)
    return jiwer.wer(refs, preds)

decoder = build_decoder(testset)
wer_fp32 = onnx_wer(FP32, testset, decoder)
wer_int8 = onnx_wer(INT8, testset, decoder)
mb = lambda p: os.path.getsize(p) / 1e6
print("========= TINYMYO (8-layer) RESULTS =========")
print(f"FP32 ONNX : WER {wer_fp32*100:6.2f}%   size {mb(FP32):6.2f} MB")
print(f"INT8 ONNX : WER {wer_int8*100:6.2f}%   size {mb(INT8):6.2f} MB   ({mb(FP32)/mb(INT8):.2f}x)")
print(f"RQ1 margin: INT8 <= {wer_fp32*1.10*100:.2f}%  -> {'PASS' if wer_int8 <= wer_fp32*1.10 else 'FAIL'}")

model_fp32_tinymyo.onnx:  10%|█         | 10/99 [00:33<06:29,  4.38s/it]

In [ ]:
OUT = f"{DRIVE_PROJECT}/onnx"; os.makedirs(OUT, exist_ok=True)
for p in [FP32, INT8]:
    shutil.copy(p, OUT); print("saved", os.path.join(OUT, os.path.basename(p)))

In [ ]:
# Find how the repo builds its decoder in --evaluate_saved
!grep -rn "ctc_decoder\|lm_weight\|word_score\|sil_score\|unk_score\|nbest\|beam_size\|beam_threshold\|lexicon\|blank_token" \
    recognition_model.py *.py config/ 2>/dev/null | head -60

In [ ]:
import jiwer, torch, torch.nn.functional as F, tqdm
model_gpu = model.to("cuda").eval()
dec = build_decoder(testset, beam=1500)
refs, preds = [], []
for i in tqdm.tqdm(range(30)):
    ex = testset[i]; x = ex["raw_emg"].unsqueeze(0).to("cuda")
    with torch.no_grad():
        logp = F.log_softmax(model_gpu(x, x, x), dim=-1).cpu()
    words = dec(logp)[0][0].words
    refs.append(testset.text_transform.clean_text(ex["text"]))
    preds.append(testset.text_transform.clean_text(" ".join(words).strip()))
print("torch + build_decoder WER (30 utts):", f"{jiwer.wer(refs,preds)*100:.2f}%")
# ~39% -> ONNX is faithful; the gap is decoder settings (fix build_decoder to match the grep)
# ~34-35% -> the ONNX export degraded it (a different fix)